# The lab's Polymarket database: a walk-through
*Database and data guide by June Qin, UCLA Trustworthy AI Lab. Questions about the course go to Prof. Cheng.*

Stats C161/C261, guest lecture demo material. Three data sources, all read-only:

1. **Catalog** (Postgres on Supabase): the `markets`, `events` and `resolutions` tables. *What markets exist, their ids, and who won.*
2. **Order books** (hourly parquet files on CloudFront): `snap-<hour>.parquet` (full book at the start of the hour) and `events-<hour>.parquet` (every level change during the hour). *What the book looked like at any moment.*
3. **Wallet trades** (daily parquet files, same host): `wallet_trades/trades-<date>.parquet`, one row per fill with a **pseudonymous** wallet. *Who traded what, without saying who they are.*

The connection string lives in a `.env` file next to this notebook and is never printed.

**Why Python?** You could browse the catalog with a database GUI, or type SQL into DuckDB's own window. But only
Python lets you look up a market, pull its data, compute on it and plot it in one place, and everything later in
the course (features, models, evaluation) happens in Python anyway. So Python is the recommended way in.

**How to read this notebook**
- **Part A, start here.** Short examples. Each one is a single question to the database, and the answer is a table. If you can change a word inside the quotes, you can use this data.
- **Part B, going further.** Rebuilding an order book tick by tick. More code, same two data sources.

# Part A. Start here
## A0. Setup: run this cell once
You do not need to understand every line. It gives you two helpers:
`ask("...")` sends a question (in SQL) to the catalog, and `book("...")` sends one to the order-book files.
Both return an ordinary pandas table.

In [ ]:
import os, json, time
import pandas as pd, duckdb
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 60)

load_dotenv()                                   # reads JUNE_DB_URL from ./.env
engine = create_engine(os.environ["JUNE_DB_URL"], connect_args={"sslmode": "require"})
BASE = "https://d1yyqbluk0oz0t.cloudfront.net"  # order-book files, public
duckdb.sql("INSTALL httpfs; LOAD httpfs;")

def ask(sql, **params):
    """Ask the catalog (Postgres) a question. Returns a table."""
    with engine.connect() as con:
        return pd.read_sql(text(sql), con, params=params or None)

def book(sql):
    """Ask the order-book files (parquet) a question. Returns a table."""
    return duckdb.sql(sql).df()

print("ready")

## A1. What does a market look like?
One market is one row of the `markets` table. `SELECT *` means "all columns", `LIMIT 1` means "just one row".
The `.T` at the end turns the row on its side so every field is readable.

In [ ]:
ask("""
    SELECT * FROM markets
    WHERE end_date > now()
    ORDER BY volume_1wk DESC NULLS LAST
    LIMIT 1
""").T

## A2. Find markets about a topic
Change `bitcoin` to anything you like. `ILIKE '%word%'` means "the title contains this word".
`end_date > now()` keeps only markets that have not ended yet (do not rely on the `active` column, see Part B).

In [ ]:
ask("""
    SELECT title, last_trade_price, volume_1wk, end_date
    FROM markets
    WHERE title ILIKE '%bitcoin%' AND end_date > now()
    ORDER BY volume_1wk DESC NULLS LAST
    LIMIT 10
""")

## A3. The ten most traded markets this week
The result is a normal pandas table, so you can keep it in a variable and plot it.

In [ ]:
top = ask("""
    SELECT title, last_trade_price, volume_1wk
    FROM markets
    WHERE end_date > now()
    ORDER BY volume_1wk DESC NULLS LAST
    LIMIT 10
""")
top

In [ ]:
import matplotlib.pyplot as plt
ax = top.set_index("title").volume_1wk.astype(float).div(1e6).sort_values().plot.barh(figsize=(10, 4))
ax.set_xlabel("volume in the last week (million $)"); ax.set_ylabel("")
ax.set_title("Ten most traded live markets"); plt.tight_layout(); plt.show()

## A4. One event, many markets
Markets belong to events. Here are all markets inside the event of the most traded market above, one per deadline.
`last_trade_price` is the market's probability. For markets whose `end_date` has passed it is the last price before they ended.

In [ ]:
event = ask("""
    SELECT event_slug FROM markets WHERE end_date > now()
    ORDER BY volume_1wk DESC NULLS LAST LIMIT 1
""").event_slug[0]
print("event:", event)
ask("""
    SELECT title, end_date, last_trade_price, volume_1wk
    FROM markets WHERE event_slug = :e
    ORDER BY end_date LIMIT 12
""", e=event)

## A5. A first statistic: what prices do live markets trade at?
A histogram of the current price of every live market with real volume. Most sit near 0: many events list dozens of possible
outcomes (every candidate, every team), and nearly all of them are long shots. A smaller pile sits near 1.

In [1]:
prices = ask("""
    SELECT last_trade_price FROM markets
    WHERE end_date > now() AND volume_1wk > 10000 AND last_trade_price IS NOT NULL
""")
ax = prices.last_trade_price.astype(float).plot.hist(bins=40, figsize=(10, 3.5))
ax.set_xlabel("last trade price of the Yes share = market probability"); ax.set_ylabel("number of markets")
ax.set_title(f"{len(prices):,} live markets with more than $10k weekly volume"); plt.tight_layout(); plt.show()

NameError: name 'ask' is not defined

## A6. Save a table to a file
One line. The CSV opens in Excel or Google Sheets.

In [ ]:
top.to_csv("top_markets.csv", index=False)
print("saved top_markets.csv with", len(top), "rows")

## A7. Peek into one hour of order-book data
The files are named by UTC hour, for example `events-2026-09-29T12.parquet`. A parquet file is just a table;
`book(...)` reads it straight from the web address. Each row says: at this time, on this side of the book,
the number of shares resting at this price became `size`.

In [ ]:
book(f"SELECT * FROM '{BASE}/events-2026-09-29T12.parquet' LIMIT 5")

In [ ]:
book(f"""
    SELECT side, count(*) AS level_changes, round(avg(price), 3) AS average_price
    FROM '{BASE}/events-2026-09-29T12.parquet'
    GROUP BY side
""")

## A8. One market's trades, grouped by wallet
The newest part of the database (October 2026): one parquet file per calendar day under `wallet_trades/`, one row per fill,
with a **pseudonymous** wallet. The same real wallet gets the same fake address in every file, so you can follow a wallet
across markets and days; you cannot look it up anywhere, and you must not try. `market` is the slug, the key into `markets`.

In [ ]:
top = book(f"""
    SELECT wallet, round(sum(dollars)) AS volume, count(*) AS trades, count(DISTINCT outcome) AS sides
    FROM '{BASE}/wallet_trades/trades-2026-10-05.parquet'
    WHERE market = 'atp-djokovi-medvede-2026-10-05'
    GROUP BY wallet ORDER BY volume DESC LIMIT 6
""")
top

Because the pseudonym is the same in every file, a wallet can be **followed**: take the biggest wallet above and ask what
else it traded in the first days of October. `read_parquet([...])` takes a list of day files and reads them in one query
(DuckDB cannot glob `*` over the web, so list the files you want).

In [ ]:
w = top.wallet[0]
days = [f"'{BASE}/wallet_trades/trades-2026-10-0{d}.parquet'" for d in range(1, 6)]
book(f"""
    SELECT market, min(time) AS first_fill, max(time) AS last_fill, count(*) AS fills, round(sum(dollars)) AS dollars
    FROM read_parquet([{", ".join(days)}])
    WHERE wallet = '{w}'
    GROUP BY market ORDER BY dollars DESC LIMIT 10
""")

In [ ]:
book(f"SELECT count(*) AS fills, count(DISTINCT wallet) AS wallets, count(DISTINCT market) AS markets, round(sum(dollars)) AS usd FROM '{BASE}/wallet_trades/trades-2026-10-03.parquet'")

That is everything you need for a first analysis: `ask` for markets and events, `book` for order books and trades,
and pandas for the rest.

---
# Part B. Going further: rebuild an order book

## Step 1. How big is the catalog?

In [ ]:
t = time.time()
counts = pd.read_sql("""
    SELECT (SELECT count(*) FROM markets) AS markets,
           (SELECT count(*) FROM events)  AS events,
           to_timestamp((SELECT max(synced_at) FROM markets)) AS last_sync
""", engine)
print(f"query took {time.time()-t:.2f} s")
counts

## Step 2. Pick an hour
Files are named by UTC hour. We use a fixed hour so that everyone gets the same result: **2026-09-29 17:00 UTC**,
a Tuesday at 10 in the morning in Los Angeles. Set `HOUR = None` to use the latest hour available instead.

In [ ]:
import datetime as dt, urllib.request
HOUR = "2026-09-29T17"                                  # fixed hour, reproducible; None = latest available
if HOUR is None:
    now = dt.datetime.now(dt.timezone.utc).replace(minute=0, second=0, microsecond=0)
    for back in range(0, 72):
        HOUR = (now - dt.timedelta(hours=back)).strftime("%Y-%m-%dT%H")
        try:
            urllib.request.urlopen(urllib.request.Request(f"{BASE}/events-{HOUR}.parquet", method="HEAD"), timeout=15)
            break
        except Exception:
            pass
print("hour:", HOUR, "UTC")

## Step 3. Which market was the busiest in that hour?
DuckDB reads the parquet file straight from the URL, no download. We count level changes per market
and join the winners back to the catalog to learn their names.

In [ ]:
t = time.time()
busiest = duckdb.sql(f"""
    SELECT market AS condition_id, count(*) AS level_changes
    FROM '{BASE}/events-{HOUR}.parquet'
    GROUP BY market ORDER BY level_changes DESC LIMIT 5
""").df()
print(f"query took {time.time()-t:.1f} s")
names = pd.read_sql("SELECT condition_id, title, slug, end_date, volume_24hr, token_ids, outcomes "
                    "FROM markets WHERE condition_id = ANY(%(ids)s)",
                    engine, params={"ids": busiest.condition_id.tolist()})
busiest = busiest.merge(names, on="condition_id")
busiest[["title", "level_changes", "volume_24hr", "end_date"]]

In [ ]:
m = busiest.iloc[0]
token_ids = m.token_ids if isinstance(m.token_ids, list) else json.loads(m.token_ids)
outcomes  = m.outcomes  if isinstance(m.outcomes, list)  else json.loads(m.outcomes)
asset = dict(zip(outcomes, token_ids))
YES = asset["Yes"]
print("market      :", m.title)
print("Yes asset id:", YES[:20] + "...")

## Step 4. Load the snapshot and the deltas, then rebuild the book
The guide's recipe: take the **snapshot** (the full book at the start of the hour), then apply every **delta**
(level change) in time order. Each delta says: at this millisecond, on this side, the number of shares at this price
became `size` (0 means the level disappeared). After each change we note the best bid and the best ask.

One detail that matters: several changes can carry the **same millisecond**, and `ORDER BY ts_ms` alone leaves their
order to chance (DuckDB reads the file in parallel). Two changes to the same price level in the wrong order give a
wrong book. So we also sort by the file's row number, which is the order the recorder wrote them in.

In [ ]:
t = time.time()
snap = duckdb.sql(f"SELECT bids, asks FROM '{BASE}/snap-{HOUR}.parquet' WHERE asset_id = '{YES}'").df()
deltas = duckdb.sql(f"""
    SELECT ts_ms, side, price, size
    FROM read_parquet('{BASE}/events-{HOUR}.parquet', file_row_number = true)
    WHERE asset_id = '{YES}'
    ORDER BY ts_ms, file_row_number          -- several changes can share a millisecond: keep the recorder's order
""").df()
seed_bids = {l["price"]: l["size"] for l in snap.bids[0]} if len(snap) else {}
seed_asks = {l["price"]: l["size"] for l in snap.asks[0]} if len(snap) else {}
print(f"loaded in {time.time()-t:.1f} s: snapshot has {len(seed_bids)} bid and {len(seed_asks)} ask levels, "
      f"{len(deltas):,} level changes in the hour")

def replay(bids, asks, deltas, start_ms=0):
    """Apply the deltas to a book; return best bid/ask after each change (from start_ms on) and the final book."""
    bids, asks, rows = dict(bids), dict(asks), []
    for ts, side, price, size in deltas.itertuples(index=False):
        book = bids if side == "BUY" else asks
        if size == 0: book.pop(price, None)
        else:         book[price] = size
        if bids and asks and ts >= start_ms:
            rows.append((ts, max(bids), min(asks)))
    path = pd.DataFrame(rows, columns=["ts_ms", "best_bid", "best_ask"])
    path["t"] = pd.to_datetime(path.ts_ms, unit="ms", utc=True)
    path["mid"] = (path.best_bid + path.best_ask) / 2
    path["spread"] = path.best_ask - path.best_bid
    return path, bids, asks

path, bids, asks = replay(seed_bids, seed_asks, deltas)
path.tail()

## Step 5. Trust, but verify
A real order book can never have a bid at or above an ask (someone would trade at once). So count the moments where
our rebuilt book is **crossed**. Zero means the snapshot and the deltas agree. If it is not zero, the snapshot is
**stale**: the recorder kept re-sending an old book. We saw exactly that on Sep 29 for the busiest market of the hour
(two identical snapshots, five cents away from the live book). The fix: after the recorder reconnects, the deltas contain
a *book reset*, a burst of hundreds of rows in one millisecond, and you can rebuild from that point instead.

In [ ]:
crossed = int((path.spread <= 0).sum())
stale = crossed > 0.01 * len(path)                      # a handful of crossed moments is timing noise; thousands is a stale snapshot
print(f"crossed-book moments: {crossed} of {len(path):,}   ->   ", "snapshot STALE" if stale else "snapshot OK")
if stale:
    per_ms = deltas.groupby("ts_ms").size()
    bursts = per_ms[per_ms >= 100]                      # a book reset = many levels in one millisecond
    start = bursts.index.min() if len(bursts) else 0    # no reset in this hour: replay from an empty book instead
    path, bids, asks = replay({}, {}, deltas, start_ms=start)
    how = f"from the book reset at {pd.to_datetime(start, unit='ms', utc=True):%H:%M:%S} UTC" if start else "from an empty book (levels untouched this hour are missing)"
    print(f"rebuilt {how}; crossed-book moments now: {int((path.spread <= 0).sum())}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].step(path.t, path.best_ask, where="post", lw=0.8, label="best ask")
ax[0].step(path.t, path.best_bid, where="post", lw=0.8, label="best bid")
ax[0].plot(path.t, path.mid, lw=1.6, color="k", label="mid = market probability")
ax[0].set_ylabel("price of a Yes share ($)"); ax[0].legend(loc="best")
ax[0].set_title(f"{m.title}\n{HOUR} UTC, rebuilt from the snapshot + {len(deltas):,} level changes")
ax[1].step(path.t, path.spread, where="post", lw=0.8, color="tab:red"); ax[1].set_ylabel("spread ($)")
plt.tight_layout(); plt.show()

## Step 6. The book at the end of the hour: the depth ladder

In [ ]:
b = pd.Series(bids).sort_index(ascending=False).head(15).sort_index()   # 15 best bids, low to high
a = pd.Series(asks).sort_index().head(15)                                # 15 best asks, low to high
fig, ax = plt.subplots(figsize=(11, 5))
ax.barh([f"{p:.3f}" for p in b.index], b.values, color="tab:green", label="bids (want to buy Yes)")
ax.barh([f"{p:.3f}" for p in a.index], a.values, color="tab:red",   label="asks (want to sell Yes)")
ax.set_xlabel("shares resting at this price"); ax.set_ylabel("price (higher is up)"); ax.legend()
ax.set_title(f"Order book at {path.t.iloc[-1]:%H:%M:%S} UTC   best bid {max(bids):.3f} | best ask {min(asks):.3f}")
plt.tight_layout(); plt.show()

## Step 7. Does the snapshot always agree? Check the eight busiest markets
The same test, run on the eight markets with the most level changes in this hour. A row that says STALE is a
market whose snapshot disagrees with its own deltas (more than 1% of the moments are crossed). A single crossed
moment is timing noise between two messages and is fine.

In [ ]:
t = time.time()
top8 = duckdb.sql(f"""
    SELECT market AS condition_id, count(*) AS level_changes
    FROM '{BASE}/events-{HOUR}.parquet' GROUP BY market ORDER BY level_changes DESC LIMIT 8
""").df()
names8 = pd.read_sql("SELECT condition_id, title, token_ids, outcomes FROM markets WHERE condition_id = ANY(%(ids)s)",
                     engine, params={"ids": top8.condition_id.tolist()})
top8 = top8.merge(names8, on="condition_id")
snap8 = duckdb.sql(f"SELECT asset_id, bids, asks FROM '{BASE}/snap-{HOUR}.parquet'").df().set_index("asset_id")
rows = []
for _, mk in top8.iterrows():
    tok = mk.token_ids if isinstance(mk.token_ids, list) else json.loads(mk.token_ids)
    out = mk.outcomes  if isinstance(mk.outcomes, list)  else json.loads(mk.outcomes)
    yes = dict(zip(out, tok))["Yes"]
    d8 = duckdb.sql(f"SELECT ts_ms, side, price, size FROM read_parquet('{BASE}/events-{HOUR}.parquet', file_row_number = true) "
                    f"WHERE asset_id = '{yes}' ORDER BY ts_ms, file_row_number").df()
    sb = {l["price"]: l["size"] for l in snap8.loc[yes, "bids"]} if yes in snap8.index else {}
    sa = {l["price"]: l["size"] for l in snap8.loc[yes, "asks"]} if yes in snap8.index else {}
    p8, b8, a8 = replay(sb, sa, d8)
    c8 = int((p8.spread <= 0).sum())
    rows.append(dict(market=mk.title[:58], level_changes=len(d8), snapshot_bid=max(sb) if sb else None, snapshot_ask=min(sa) if sa else None,
                     book_end_bid=max(b8) if b8 else None, book_end_ask=min(a8) if a8 else None,
                     crossed=c8, verdict="STALE" if c8 > 0.01 * len(p8) else "OK"))
print(f"checked in {time.time()-t:.0f} s")
pd.DataFrame(rows)

## Step 8. What else the database holds: who won
`resolutions` has one row per resolved market since Jan 25 2026: `resolved_yes`, `price_at_close`, `resolved_at`. New rows arrive
about 15 minutes after a market closes. Together with the wallet-trade files (A8) this is the ground truth for
"did this wallet buy the winning side, and how long before the close". Two things it does not have yet: `price_24h_before`
is empty, and `market_hourly` is an empty table. Note that `resolved_at` is the real close; `end_date` in `markets` is only the scheduled one.

In [ ]:
ask("""
    SELECT count(*) AS resolved_markets, count(*) FILTER (WHERE resolved_yes) AS yes_won,
           min(resolved_at) AS first, max(resolved_at) AS last, count(price_24h_before) AS price_24h_filled
    FROM resolutions
""")

In [ ]:
ask("""
    SELECT m.title, r.resolved_yes, r.price_at_close, r.resolved_at, m.end_date AS scheduled_end
    FROM resolutions r JOIN markets m ON m.slug = r.market_slug
    ORDER BY r.resolved_at DESC LIMIT 5
""")

## Step 9. One more catch: the status flags are never updated
`active = true` for every market, so filter on `end_date` instead.

In [ ]:
pd.read_sql("""
    SELECT count(*) AS total,
           count(*) FILTER (WHERE closed) AS flagged_closed,
           count(*) FILTER (WHERE end_date < now()) AS past_end_date
    FROM markets
""", engine)